# Study DESC shear matching

Analyzes the DESC metadetect shear match: SNR distributions, catalog coverage, and spatial distributions of good vs bad matches.

In [ ]:
# Standard imports
import matplotlib.pyplot as plt
import numpy as np
import tables_io
import hpmcm

## Configuration

In [ ]:
tract = 8006

In [ ]:
# Load DESC shear match output tables and input catalog files
shear_desc_in = tables_io.read(f'{tract}/mdet-run-{tract}.parq')
shear_desc_ns = tables_io.read(f'{tract}/mdet-run-reduced_uncleaned_{tract}_ns.parq')
shear_desc_1m = tables_io.read(f'{tract}/mdet-run-reduced_uncleaned_{tract}_1m.parq')
shear_desc_1p = tables_io.read(f'{tract}/mdet-run-reduced_uncleaned_{tract}_1p.parq')
shear_desc_obj_stats = tables_io.read(f'{tract}/desc_md_match_object_stats.pq')
shear_desc_obj_assoc = tables_io.read(f'{tract}/desc_md_match_object_assoc.pq')
input_files = [
    f'{tract}/mdet-run-reduced_uncleaned_{tract}_ns.parq',
    f'{tract}/mdet-run-reduced_uncleaned_{tract}_1m.parq',
    f'{tract}/mdet-run-reduced_uncleaned_{tract}_1p.parq',
]    
catalog_ids = [0, 1, 2]

In [ ]:
# Build wide joined table — one row per object with all catalog columns
joined_shear_obj = hpmcm.output_tables.buildJoinedObjectTable(
    shear_desc_obj_stats,
    shear_desc_obj_assoc,
    input_files,
    catalog_ids,
)

In [ ]:
# SNR histograms split by catalog detection bits
bins = np.logspace(0, 5, 51)
for i in range(3):
    mask_bit = 1 << i
    mask = np.bitwise_and(joined_shear_obj.catalog_mask, mask_bit) == mask_bit
    _ = plt.hist(joined_shear_obj[mask].snr, bins=bins, alpha=0.1)
_ = plt.xscale('log')
_ = plt.yscale('log')

In [ ]:
_ = plt.hist2d(joined_shear_obj.catalog_mask, joined_shear_obj.snr, bins=(np.linspace(0.5, 7.5, 8), np.logspace(0, 5, 51)), norm='log')
_ = plt.yscale('log')

In [ ]:
# Objects detected in all 3 DESC catalogs (ns, 1m, 1p) have catalog_mask == 7
good_mask = joined_shear_obj.catalog_mask == 7
bad_mask = joined_shear_obj.catalog_mask != 7
good_sub = joined_shear_obj[good_mask]
bad_sub = joined_shear_obj[bad_mask]

In [ ]:
# Spatial distribution of good/bad matches in cell 1
_ = plt.scatter(good_sub.x_cell_coadd_1[::10], good_sub.y_cell_coadd_1[::10], s=1)
_ = plt.scatter(bad_sub.x_cell_coadd_1[::5], bad_sub.y_cell_coadd_1[::5], s=1)

In [ ]:
_ = plt.scatter(good_sub.x_cell_coadd_2[::10], good_sub.y_cell_coadd_2[::10], s=1)
_ = plt.scatter(bad_sub.x_cell_coadd_2[::5], bad_sub.y_cell_coadd_2[::5], s=1)

In [ ]:
_ = plt.scatter(good_sub.x_pix[::5], good_sub.y_pix[::5], s=1)
_ = plt.scatter(bad_sub.x_pix[::5], bad_sub.y_pix[::5], s=1)